# Hidden Markov Model (HMM) Analysis
### Master Thesis: Revisiting Delegation Theory in the Age of AI: Dynamic Algorithm Appreciation and Aversion in Triadic Organizational Relationships

---

### Table of Contents

**Part I — Data and HMM Estimation Framework**

1. [Data Description and Variable Construction](#1-data-description-and-variable-construction)
   - 1.1 &nbsp; [Helpers & Imports](#11-helpers--imports)
   - 1.2 &nbsp; [Benchmark Construction](#12-benchmark-construction)
   - 1.3 &nbsp; [Data Loading & Inspection](#13-data-loading--inspection)
2. [Hidden Markov Model Specification](#2-hidden-markov-model-specification)
   - 2.1 &nbsp; [Model Structure & Forward–Backward Algorithm](#21-model-structure--22-forwardbackward-algorithm)
   - 2.2 &nbsp; [Maximum Likelihood Estimation](#22-maximum-likelihood-estimation)
3. [Model Selection and Parameter Estimation](#3-model-selection-and-parameter-estimation)
4. [Posterior State Inference and Interpretation](#4-posterior-state-inference-and-interpretation)
   - 4.1 &nbsp; [Posterior State Assignment](#41-posterior-state-assignment)
   - 4.2 &nbsp; [Results Visualisation](#42-results-visualisation)

**Part II — Empirical Findings**

5. [KPI-Driven Transition Dynamics](#5-kpi-driven-transition-dynamics)
6. [Transparency as Moderator of State Transitions](#6-transparency-as-moderator-of-state-transitions)
7. [Strategic Control Retention Under High Task Stakes](#7-strategic-control-retention-under-high-task-stakes)

**Part III — Theoretical and Task-Level Implications**

8. [Delegation as Dynamic Learning Process](#8-delegation-as-dynamic-learning-process)
9. [Authority as Strategic and Legitimacy Mechanism](#9-authority-as-strategic-and-legitimacy-mechanism)
10. [Triadic Delegation Flows: Authority vs Execution](#10-triadic-delegation-flows-authority-vs-execution)
11. [Temporal Evolution of Delegation Patterns](#11-temporal-evolution-of-delegation-patterns)

**Appendix**

12. [Cluster Bootstrap Standard Errors](#12-cluster-bootstrap-standard-errors)
13. [Model Summary Table](#13-model-summary-table)

---
# Part I — Data and HMM Estimation Framework

<a id="1-data-description-and-variable-construction"></a>
## 1. Data Description and Variable Construction
<a id="11-helpers--imports"></a>
### 1.1 Helpers & Imports

In [1]:
# ============================================================
# 1. Imports & Helpers
# ============================================================
from __future__ import annotations

import os
import time
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple
from pathlib import Path
from multiprocessing.pool import ThreadPool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

# ---- resolve data path ----
MANUAL_XLSX_PATH = None
PREFERRED_DATASETS = [
    Path(r"..\new run\Triadic_Delegation_Dataset_SYNTH_ANALYSIS_v2.xlsx"),
]
DATA_PATH = None
if MANUAL_XLSX_PATH:
    DATA_PATH = Path(MANUAL_XLSX_PATH)
else:
    for p in PREFERRED_DATASETS:
        if p.exists():
            DATA_PATH = p
            break
assert DATA_PATH is not None and DATA_PATH.exists(), \
    f"Data file not found. Tried: {PREFERRED_DATASETS}"
print(f"Data file: {DATA_PATH.resolve()}")


def softmax(z, axis=-1):
    """Numerically stable softmax (works on 1-D vectors and row-wise on 2-D)."""
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z)
    return e / np.sum(e, axis=axis, keepdims=True)


def log_softmax(z, axis=-1):
    """Numerically stable log-softmax."""
    return z - logsumexp(z, axis=axis, keepdims=True)


def log_gaussian_diag(y, mean, log_sigma):
    """Scalar version (kept for reference)."""
    sigma2 = np.exp(2 * log_sigma)
    return -0.5 * (
        np.sum(np.log(2 * np.pi * sigma2))
        + np.sum((y - mean) ** 2 / sigma2)
    )

print("Section 1 — Imports & helpers loaded.")

Data file: C:\Users\Admin\OneDrive\Desktop\Algorithm-Appreciation-and-Aversion-in-Triadic-Delegation-Settings\new run\Triadic_Delegation_Dataset_SYNTH_ANALYSIS_v2.xlsx
Section 1 — Imports & helpers loaded.


<a id="12-benchmark-construction"></a>
### 1.2 Benchmark Construction
Derive performance benchmarks used as **transition covariates** (centered specification):
- `kpi_operational_gap_index` — composite operational KPIs gap (intermediate, used to build benchmarks)
- `within_unit_temporal_benchmark_c` — centered period-over-period KPI change
- `horizontal_peer_benchmark_c` — centered cross-sectional percentile rank within period
- `threshold_benchmark_c` — centered recent negative shock indicator
- `transparency_level_norm` — transparency main effect (normalised 0–1)
- `transparency_x_temporal_c` — transparency × centered temporal benchmark
- `transparency_x_peer_c` — transparency × centered peer benchmark
- `transparency_x_threshold_c` — transparency × centered threshold shock

> `within_unit_ai_trajectory_benchmark` remains trimmed due to multicollinearity (high VIF).

In [15]:
# ============================================================
# 2. Build Benchmarks
# ============================================================

def build_benchmarks(df):
    df = df.sort_values(["manager_id", "period_id"]).copy()

    # Composite performance index (higher = better)
    df["kpi_operational_gap_index"] = (
        (-1.0) * df["service_level_delta"]
        + (-0.6) * df["inventory_cost_delta"]
        + (-0.4) * df["expedite_cost_delta"]
        + (-1.2) * df["error_incident_count"]
    )

    # Within-manager temporal benchmark (period-over-period diff)
    df["within_unit_temporal_benchmark"] = (
        df.groupby("manager_id")["kpi_operational_gap_index"].diff(1)
    )

    # Peer percentile (cross-sectional)
    pct = df.groupby("period_id")["kpi_operational_gap_index"].rank(pct=True)
    df["horizontal_peer_benchmark"] = (pct - 0.5) * 2.0

    # Threshold shock (binary)
    df["threshold_benchmark"] = df["recent_negative_shock"].astype(float)

    # Mean-center the three benchmark variables
    df["within_unit_temporal_benchmark_c"] = (
        df["within_unit_temporal_benchmark"]
        - df["within_unit_temporal_benchmark"].mean()
    )
    df["horizontal_peer_benchmark_c"] = (
        df["horizontal_peer_benchmark"]
        - df["horizontal_peer_benchmark"].mean()
    )
    df["threshold_benchmark_c"] = (
        df["threshold_benchmark"]
        - df["threshold_benchmark"].mean()
    )

    # Transparency main effect
    df["transparency_level_norm"] = df["transparency_level"].astype(float) / 3.0

    # Transparency × centered KPI benchmark interactions
    df["transparency_x_temporal_c"] = (
        df["transparency_level_norm"] * df["within_unit_temporal_benchmark_c"]
    )
    df["transparency_x_peer_c"] = (
        df["transparency_level_norm"] * df["horizontal_peer_benchmark_c"]
    )
    df["transparency_x_threshold_c"] = (
        df["transparency_level_norm"] * df["threshold_benchmark_c"]
    )

    return df

print("Section 2 — build_benchmarks() defined with centered benchmarks + centered interactions.")

Section 2 — build_benchmarks() defined with centered benchmarks + centered interactions.


<a id="13-data-loading--inspection"></a>
### 1.3 Data Loading & Inspection
Load the `panel_manager_period` sheet, build benchmarks, scale variables, and form per-manager sequences.

In [16]:
# ============================================================
# 3. Data Loader
# ============================================================

@dataclass
class HMMData:
    Y: List[np.ndarray]      # emission sequences
    X: List[np.ndarray]      # transition covariates
    Z: List[np.ndarray]      # emission controls
    ids: List[str]
    periods: List[np.ndarray]
    y_scaler: StandardScaler
    x_scaler: StandardScaler
    z_scaler: StandardScaler


def load_sequences(xlsx_path, transition_cols_override: Optional[List[str]] = None):
    # Cache raw sheets to avoid repeated Excel I/O in extension screening
    panel_cache_key = "_panel_manager_period_cache"
    decision_cache_key = "_decision_episode_cache"

    if panel_cache_key not in globals():
        globals()[panel_cache_key] = pd.read_excel(xlsx_path, sheet_name="panel_manager_period")
    if decision_cache_key not in globals():
        globals()[decision_cache_key] = pd.read_excel(xlsx_path, sheet_name="decision_episode")

    df = globals()[panel_cache_key].copy()

    # Safety: analysis file should NOT contain latent truth columns
    forbidden = ["latent_state_true", "latent_state_true_next"]
    if any(c in df.columns for c in forbidden):
        df = df.drop(columns=[c for c in forbidden if c in df.columns])
        print("  Dropped latent truth columns to proceed with analysis data.")

    df = build_benchmarks(df)

    # ── Compute share_authority_esc from decision_episode ──
    dec_ep = globals()[decision_cache_key].copy()
    dec_ep["is_escalated"] = (dec_ep["escalation_flag"] == 1).astype(int)
    esc_agg = (dec_ep
        .groupby(["manager_id", "period_id"])
        .agg(n_tasks=("episode_id", "count"),
             n_escalated=("is_escalated", "sum"))
        .reset_index())
    esc_agg["share_authority_esc"] = esc_agg["n_escalated"] / esc_agg["n_tasks"]
    df = df.merge(esc_agg[["manager_id", "period_id", "share_authority_esc"]],
                  on=["manager_id", "period_id"], how="left")
    df["share_authority_esc"] = df["share_authority_esc"].fillna(0.0)
    print(f"  share_authority_esc computed from decision_episode "
          f"(mean={df['share_authority_esc'].mean():.4f}, "
          f"std={df['share_authority_esc'].std():.4f})")

    # D=2: AI authority share + manager escalation share
    emission_cols = ["ai_decision_authority_share", "share_authority_esc"]

    # Default P=7: centered benchmarks + transparency + centered interactions
    transition_cols_default = [
        "within_unit_temporal_benchmark_c",
        "horizontal_peer_benchmark_c",
        "threshold_benchmark_c",
        "transparency_level_norm",
        "transparency_x_temporal_c",
        "transparency_x_peer_c",
        "transparency_x_threshold_c",
    ]
    transition_cols = transition_cols_override if transition_cols_override is not None else transition_cols_default

    control_cols = [
        "task_complexity_index",
        "demand_volatility",
        "supply_disruption_count",
        "forecast_accuracy_mape",
        "decision_latency_avg",
        "target_difficulty",
        "performance_pressure_index",
        "recent_negative_shock",
    ]

    df = df.dropna(subset=emission_cols + transition_cols + control_cols)

    Y_list, X_list, Z_list = [], [], []
    ids, periods = [], []

    for mid, g in df.groupby("manager_id"):
        g = g.sort_values("period_id")
        Y = g[emission_cols].to_numpy(float)
        X = g[transition_cols].to_numpy(float)
        Z = g[control_cols].to_numpy(float)
        if len(Y) < 3:
            continue
        Y_list.append(Y)
        X_list.append(X)
        Z_list.append(Z)
        ids.append(mid)
        periods.append(g["period_id"].to_numpy())

    y_scaler = StandardScaler().fit(np.vstack(Y_list))
    x_scaler = StandardScaler().fit(np.vstack(X_list))
    z_scaler = StandardScaler().fit(np.vstack(Z_list))

    Y_list = [y_scaler.transform(y) for y in Y_list]
    X_list = [x_scaler.transform(x) for x in X_list]
    Z_list = [z_scaler.transform(z) for z in Z_list]

    return HMMData(Y_list, X_list, Z_list, ids, periods,
                   y_scaler, x_scaler, z_scaler)


# ---- Load & inspect ----
data = load_sequences(DATA_PATH)

seq_lens = [len(y) for y in data.Y]
print(f"Managers loaded : {len(data.Y)}")
print(f"Total observations: {sum(seq_lens)}")
print(f"Sequence lengths : min={min(seq_lens)}, median={int(np.median(seq_lens))}, max={max(seq_lens)}")
print(f"Emission dims (D) : {data.Y[0].shape[1]}")
print(f"Trans. covars (P) : {data.X[0].shape[1]}  "
      f"(centered benchmarks + transparency + centered interactions)")
print(f"Controls (K)      : {data.Z[0].shape[1]}")

  share_authority_esc computed from decision_episode (mean=0.3508, std=0.0731)
Managers loaded : 120
Total observations: 3000
Sequence lengths : min=25, median=25, max=25
Emission dims (D) : 2
Trans. covars (P) : 7  (centered benchmarks + transparency + centered interactions)
Controls (K)      : 8


<a id="2-hidden-markov-model-specification"></a>
## 2. Hidden Markov Model Specification
<a id="21-model-structure--22-forwardbackward-algorithm"></a>
### 2.1 Model Structure & 2.2 Forward–Backward Algorithm

In [17]:
# ============================================================
# Pre-stack all sequences to 3-D tensors for batched estimation
# ============================================================
import numpy as np

Y_stack = np.stack(data.Y)   # (N, T, D)
X_stack = np.stack(data.X)   # (N, T, P)
Z_stack = np.stack(data.Z)   # (N, T, K)

N, T, D = Y_stack.shape
P = X_stack.shape[2]
K = Z_stack.shape[2]
n_obs_total = N * T

print(f"Stacked arrays:")
print(f"  Y_stack: {Y_stack.shape}  (N={N}, T={T}, D={D})")
print(f"  X_stack: {X_stack.shape}  (N={N}, T={T}, P={P})")
print(f"  Z_stack: {Z_stack.shape}  (N={N}, T={T}, K={K})")
print(f"  n_obs_total = {n_obs_total}")

Stacked arrays:
  Y_stack: (120, 25, 2)  (N=120, T=25, D=2)
  X_stack: (120, 25, 7)  (N=120, T=25, P=7)
  Z_stack: (120, 25, 8)  (N=120, T=25, K=8)
  n_obs_total = 3000


In [18]:
# ── X collinearity check (centered benchmark design) ───────────────────────────
transition_cols_check = [
    "within_unit_temporal_benchmark_c",
    "horizontal_peer_benchmark_c",
    "threshold_benchmark_c",
    "transparency_level_norm",
    "transparency_x_temporal_c",
    "transparency_x_peer_c",
    "transparency_x_threshold_c",
]

X_flat = X_stack.reshape(-1, X_stack.shape[-1])   # (N*T, P)
corr = np.corrcoef(X_flat.T)
df_corr = pd.DataFrame(corr, index=transition_cols_check, columns=transition_cols_check)

def color_high(val):
    return "background-color: #ff4444; color: white" if abs(val) >= 0.80 and abs(val) < 1.0 else ""

display(df_corr.round(3).style.map(color_high))

print("\nPairs with |r| >= 0.70:")
found = False
for i in range(len(transition_cols_check)):
    for j in range(i+1, len(transition_cols_check)):
        r = corr[i, j]
        if abs(r) >= 0.70:
            print(f"  {transition_cols_check[i]:40s}  {transition_cols_check[j]:40s}  r={r:.3f}")
            found = True
if not found:
    print("  None — all pairs below 0.70 ✓")

,within_unit_temporal_benchmark_c,horizontal_peer_benchmark_c,threshold_benchmark_c,transparency_level_norm,transparency_x_temporal_c,transparency_x_peer_c,transparency_x_threshold_c
within_unit_temporal_benchmark_c,1.000000,0.446000,-0.473000,-0.000000,0.856000,0.362000,-0.399000
horizontal_peer_benchmark_c,0.446000,1.000000,-0.521000,-0.000000,0.376000,0.860000,-0.454000
threshold_benchmark_c,-0.473000,-0.521000,1.000000,0.010000,-0.416000,-0.457000,0.865000
transparency_level_norm,-0.000000,-0.000000,0.010000,1.000000,-0.029000,0.000000,0.005000
transparency_x_temporal_c,0.856000,0.376000,-0.416000,-0.029000,1.000000,0.434000,-0.480000
transparency_x_peer_c,0.362000,0.860000,-0.457000,0.000000,0.434000,1.000000,-0.531000
transparency_x_threshold_c,-0.399000,-0.454000,0.865000,0.005000,-0.480000,-0.531000,1.000000



Pairs with |r| >= 0.70:
  within_unit_temporal_benchmark_c          transparency_x_temporal_c                 r=0.856
  horizontal_peer_benchmark_c               transparency_x_peer_c                     r=0.860
  threshold_benchmark_c                     transparency_x_threshold_c                r=0.865


In [19]:
# ============================================================
# 4. Parameters + Forward–Backward  (VECTORIZED)
# ============================================================

@dataclass
class Params:
    logit_pi: np.ndarray   # (J,)
    alpha: np.ndarray      # (J, J)
    beta: np.ndarray       # (J, J, P)
    mu: np.ndarray         # (J, D)
    W: np.ndarray          # (J, D, K)
    log_sigma: np.ndarray  # (J, D)


def _precompute(p, Y, X, Z):
    """Shared emission + transition pre-computation."""
    T, D = Y.shape
    J = p.mu.shape[0]
    means = p.mu[None, :, :] + np.einsum('jdk,tk->tjd', p.W, Z)
    residuals = Y[:, None, :] - means
    sigma2 = np.exp(2 * p.log_sigma)
    log_norm = np.sum(np.log(2 * np.pi * sigma2), axis=1)
    logB = -0.5 * (log_norm[None, :] +
                   np.sum(residuals ** 2 / sigma2[None, :, :], axis=2))
    logits_all = (p.alpha[None, :, :]
                  + np.einsum('ijp,tp->tij', p.beta, X))
    logQ_all = log_softmax(logits_all, axis=2)
    return T, J, logB, logQ_all


def forward_only(p, Y, X, Z):
    """Forward pass only — returns log-likelihood (no posterior). ~2× faster."""
    T, J, logB, logQ_all = _precompute(p, Y, X, Z)
    pi = softmax(p.logit_pi)
    log_alpha = np.empty((T, J))
    log_alpha[0] = np.log(pi) + logB[0]
    for t in range(1, T):
        log_alpha[t] = logB[t] + logsumexp(
            log_alpha[t - 1, :, None] + logQ_all[t], axis=0)
    return float(logsumexp(log_alpha[-1]))


def forward_backward(p, Y, X, Z):
    """Full forward–backward returning (ll, log_gamma)."""
    T, J, logB, logQ_all = _precompute(p, Y, X, Z)
    pi = softmax(p.logit_pi)
    log_alpha = np.empty((T, J))
    log_alpha[0] = np.log(pi) + logB[0]
    for t in range(1, T):
        log_alpha[t] = logB[t] + logsumexp(
            log_alpha[t - 1, :, None] + logQ_all[t], axis=0)
    ll = logsumexp(log_alpha[-1])

    log_beta = np.zeros((T, J))
    for t in reversed(range(T - 1)):
        log_beta[t] = logsumexp(
            logQ_all[t + 1] + logB[t + 1][None, :] + log_beta[t + 1][None, :],
            axis=1)

    log_gamma = log_alpha + log_beta
    log_gamma -= logsumexp(log_gamma, axis=1, keepdims=True)
    return ll, log_gamma

print("Section 4 — Params, forward_only() & forward_backward() defined.")

Section 4 — Params, forward_only() & forward_backward() defined.


<a id="22-maximum-likelihood-estimation"></a>
### 2.2 Maximum Likelihood Estimation
Estimate models with 2–4 latent states via maximum likelihood using L-BFGS-B optimization. To mitigate local optima, we employ multiple random initializations and diagonal-biased transition logits. Model selection follows a two-stage procedure: a computationally efficient screening phase identifies the most promising state specification using BIC, followed by a high-precision refit of the selected model using extended iterations and warm-start initialization. The final model is chosen based on the lowest Bayesian Information Criterion.

In [20]:
# ============================================================
# 5. Batched MLE Estimator  (fit_model_batched)
# ============================================================
# Accepts Y_stack, X_stack, Z_stack as EXPLICIT parameters so
# that Stack dimensions are always consistent with what was passed.
# Supports do_emission_only_warmstart for a two-phase init.
# ============================================================

import time
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp, log_softmax


def fit_model_batched(
    J: int,
    Y_stack: np.ndarray,
    X_stack: np.ndarray,
    Z_stack: np.ndarray,
    *,
    maxiter: int = 600,
    n_starts: int = 5,
    seed: int = 7,
    sigma_min: float = 0.05,
    sigma_max: float = 5.0,
    time_cap_min: int = 15,
    print_every: int = 50,
    l2: float = 1e-4,
    diag_bias: float = 2.0,
    maxfun: int = 200_000,
    ftol: float = 1e-8,
    gtol: float = 5e-6,
    warm_starts: list | None = None,
    use_subset: bool = False,
    subset_size: int = 80,
    do_emission_only_warmstart: bool = True,
    emission_only_maxiter: int = 120,
    emission_only_maxfun: int = 60_000,
):
    """
    Batched NH-HMM fit with Gaussian emissions and covariate-dependent
    transitions via L-BFGS-B.

    Parameters
    ----------
    Y_stack : (N, T, D) emission data
    X_stack : (N, T, P) transition covariates
    Z_stack : (N, T, K) emission controls
    do_emission_only_warmstart : if True, first optimize emission params only
        (mu, W, log_sigma) with transitions fixed, then use as init.
    """
    # Derive dimensions from passed stacks
    N_full, T, D = Y_stack.shape
    P = X_stack.shape[2]
    K = Z_stack.shape[2]

    # Optional subset for speed
    if use_subset and N_full > subset_size:
        rng_sub = np.random.default_rng(seed)
        idx = rng_sub.choice(N_full, size=subset_size, replace=False)
        Y_use = Y_stack[idx]
        X_use = X_stack[idx]
        Z_use = Z_stack[idx]
        N_use = subset_size
    else:
        Y_use, X_use, Z_use = Y_stack, X_stack, Z_stack
        N_use = N_full

    # ── pack / unpack ──
    def pack(p):
        return np.concatenate([
            p.logit_pi.ravel(), p.alpha.ravel(), p.beta.ravel(),
            p.mu.ravel(), p.W.ravel(), p.log_sigma.ravel(),
        ])

    def unpack(theta):
        idx = 0
        def take(n):
            nonlocal idx; v = theta[idx:idx+n]; idx += n; return v
        return Params(
            logit_pi=take(J),
            alpha=take(J*J).reshape(J,J),
            beta=take(J*J*P).reshape(J,J,P),
            mu=take(J*D).reshape(J,D),
            W=take(J*D*K).reshape(J,D,K),
            log_sigma=take(J*D).reshape(J,D),
        )

    # ── bounds (log_sigma only) ──
    n_logit_pi = J
    n_alpha = J*J
    n_beta = J*J*P
    n_mu = J*D
    n_W = J*D*K
    n_log_sigma = J*D
    log_sigma_start = n_logit_pi + n_alpha + n_beta + n_mu + n_W
    log_sigma_end = log_sigma_start + n_log_sigma
    total_params = log_sigma_end

    LOW, HIGH = np.log(sigma_min), np.log(sigma_max)
    bounds = [(None, None)] * total_params
    for i in range(log_sigma_start, log_sigma_end):
        bounds[i] = (LOW, HIGH)

    # ── smart init ──
    Y_flat = Y_use.reshape(-1, D)
    y_mean = Y_flat.mean(axis=0)
    y_std = np.maximum(Y_flat.std(axis=0), 1e-3)

    def smart_init_params(rng):
        mu0 = y_mean[None,:] + rng.normal(0, 1.0, (J,D)) * y_std[None,:]
        log_sigma0 = np.log(np.clip(y_std, sigma_min, sigma_max))[None,:]
        log_sigma0 = np.repeat(log_sigma0, J, axis=0)
        log_sigma0 = np.clip(log_sigma0 + rng.normal(0, 0.12, (J,D)), LOW, HIGH)
        logit_pi0 = rng.normal(0, 0.2, J)
        alpha0 = rng.normal(0, 0.20, (J,J)) + np.eye(J) * diag_bias
        beta0 = rng.normal(0, 0.02, (J,J,P))
        W0 = rng.normal(0, 0.03, (J,D,K))
        return Params(logit_pi=logit_pi0, alpha=alpha0, beta=beta0,
                      mu=mu0, W=W0, log_sigma=log_sigma0)

    # ── neg-LL (batched forward algorithm) ──
    stop_flag = {"stop": False}

    def neg_ll(theta):
        if stop_flag["stop"]:
            return 1e50
        p = unpack(theta)
        log_pi = log_softmax(p.logit_pi, axis=0)
        means = p.mu[None,None,:,:] + np.einsum("jdk,ntk->ntjd", p.W, Z_use)
        resid = Y_use[:,:,None,:] - means
        sigma2 = np.maximum(np.exp(2.0 * p.log_sigma), 1e-6)
        log_norm = np.sum(np.log(2*np.pi * sigma2), axis=1)
        logB = -0.5 * (log_norm[None,None,:] +
                       np.sum(resid**2 / sigma2[None,None,:,:], axis=3))
        logQ = log_softmax(
            p.alpha[None,None,:,:] + np.einsum("ijp,ntp->ntij", p.beta, X_use),
            axis=3)
        la = log_pi[None,:] + logB[:,0,:]
        for t in range(1, T):
            la = logB[:,t,:] + logsumexp(la[:,:,None] + logQ[:,t,:,:], axis=1)
        ll = np.sum(logsumexp(la, axis=1))
        return -float(ll) if np.isfinite(ll) else 1e40

    def objective(theta):
        base = neg_ll(theta)
        if not np.isfinite(base) or l2 <= 0:
            return base if np.isfinite(base) else 1e40
        p = unpack(theta)
        pen = (np.sum(p.alpha**2) + np.sum(p.beta**2) +
               np.sum(p.W**2) + 0.10*np.sum(p.mu**2))
        return base + l2 * pen

    # ── emission-only warmstart objective ──
    def emission_only_objective(em_theta, fixed_logit_pi, fixed_alpha, fixed_beta):
        """Optimize only mu, W, log_sigma with transitions frozen."""
        idx = 0
        def take(n):
            nonlocal idx; v = em_theta[idx:idx+n]; idx += n; return v
        mu = take(J*D).reshape(J,D)
        W = take(J*D*K).reshape(J,D,K)
        log_sigma = take(J*D).reshape(J,D)
        p = Params(logit_pi=fixed_logit_pi, alpha=fixed_alpha,
                   beta=fixed_beta, mu=mu, W=W, log_sigma=log_sigma)
        theta_full = pack(p)
        return objective(theta_full)

    # ── multi-start optimization ──
    init_list = list(warm_starts) if warm_starts else []
    runs = []

    for s in range(n_starts):
        rng = np.random.default_rng(seed + s)
        stop_flag["stop"] = False

        # Initialize
        if s < len(init_list):
            p0 = init_list[s]
            # Ensure dimensions match
            if p0.W.shape != (J, D, K):
                print(f"  ⚠️  start {s+1}: warm start W shape {p0.W.shape} "
                      f"!= ({J},{D},{K}), using random init")
                p0 = smart_init_params(rng)
            else:
                p0 = Params(
                    logit_pi=p0.logit_pi + rng.normal(0, 0.03, J),
                    alpha=p0.alpha + rng.normal(0, 0.03, (J,J)),
                    beta=p0.beta + rng.normal(0, 0.008, (J,J,P)),
                    mu=p0.mu + rng.normal(0, 0.05, (J,D)),
                    W=p0.W + rng.normal(0, 0.01, (J,D,K)),
                    log_sigma=np.clip(p0.log_sigma + rng.normal(0, 0.02, (J,D)),
                                      LOW, HIGH),
                )
        else:
            p0 = smart_init_params(rng)

        # Phase 1 (optional): emission-only warmstart
        if do_emission_only_warmstart and s >= len(init_list):
            em_theta0 = np.concatenate([
                p0.mu.ravel(), p0.W.ravel(), p0.log_sigma.ravel()])
            em_bounds = ([(None,None)]*(J*D + J*D*K) +
                         [(LOW,HIGH)]*(J*D))
            em_res = minimize(
                emission_only_objective, em_theta0,
                args=(p0.logit_pi, p0.alpha, p0.beta),
                method="L-BFGS-B", bounds=em_bounds,
                options={"maxiter": emission_only_maxiter,
                         "maxfun": emission_only_maxfun})
            # unpack emission params back
            eidx = 0
            def etake(n):
                nonlocal eidx; v = em_res.x[eidx:eidx+n]; eidx += n; return v
            p0 = Params(logit_pi=p0.logit_pi, alpha=p0.alpha,
                        beta=p0.beta,
                        mu=etake(J*D).reshape(J,D),
                        W=etake(J*D*K).reshape(J,D,K),
                        log_sigma=etake(J*D).reshape(J,D))

        # Phase 2: full optimization
        theta0 = pack(p0)
        start_time = time.time()
        iter_counter = {"i": 0}

        def callback(_xk):
            iter_counter["i"] += 1
            if iter_counter["i"] % print_every == 0:
                elapsed_min = (time.time() - start_time) / 60
                print(f"    J={J} start {s+1}/{n_starts} "
                      f"iter={iter_counter['i']} elapsed={elapsed_min:.1f} min",
                      flush=True)
            if (time.time() - start_time) > time_cap_min * 60:
                stop_flag["stop"] = True

        res = minimize(objective, theta0, method="L-BFGS-B",
                       bounds=bounds, callback=callback,
                       options={"maxiter": maxiter, "maxfun": maxfun,
                                "ftol": ftol, "gtol": gtol})

        if stop_flag["stop"]:
            res.success = False
            res.message = f"Time cap reached ({time_cap_min} min)"

        true_negll = neg_ll(res.x)
        runs.append((unpack(res.x), res, true_negll))
        print(f"    done: J={J} start {s+1}/{n_starts} success={res.success} "
              f"nit={getattr(res,'nit',None)} true_negLL={true_negll:.2f} "
              f"msg={res.message}", flush=True)

    # ── pick best run ──
    converged = [(p,r,tnl) for (p,r,tnl) in runs if bool(r.success)]
    if converged:
        best_p, best_res, best_tnl = min(converged, key=lambda t: t[2])
        best_is_conv = True
    else:
        best_p, best_res, best_tnl = min(runs, key=lambda t: t[2])
        best_is_conv = False

    best_res.true_negll = best_tnl
    best_res.true_ll = -best_tnl
    best_res.k_params = len(best_res.x)
    return best_p, best_res, best_is_conv


print("fit_model_batched() defined — accepts Y_stack, X_stack, Z_stack explicitly.")

fit_model_batched() defined — accepts Y_stack, X_stack, Z_stack explicitly.


### 2.3 Interaction Extensions (one-at-a-time)
Run centered interaction extensions separately (Model A/B/C), compare `J=2` and `J=3`, and only run `J=4` when needed.

In [21]:
# ============================================================
# One-interaction-at-a-time extension screening (A/B/C)
# ============================================================

import numpy as np
import pandas as pd

BASE_COLS = [
    "within_unit_temporal_benchmark_c",
    "horizontal_peer_benchmark_c",
    "threshold_benchmark_c",
    "transparency_level_norm",
]

MODEL_SPECS = {
    "Baseline": BASE_COLS,
    "Model A": BASE_COLS + ["transparency_x_temporal_c"],
    "Model B": BASE_COLS + ["transparency_x_peer_c"],
    "Model C": BASE_COLS + ["transparency_x_threshold_c"],
}

J_MIN_SET = [2, 3]


def fit_and_score_model(model_name, transition_cols, J, seed):
    data_m = load_sequences(DATA_PATH, transition_cols_override=transition_cols)
    Y_m = np.stack(data_m.Y)
    X_m = np.stack(data_m.X)
    Z_m = np.stack(data_m.Z)
    n_obs_m = Y_m.shape[0] * Y_m.shape[1]

    # lightweight screening settings for A/B/C comparisons
    cfg = dict(
        maxiter=300 if J <= 3 else 450,
        n_starts=2 if J <= 3 else 2,
        time_cap_min=8 if J <= 3 else 12,
        diag_bias=2.3 if J <= 3 else 2.6,
        maxfun=180_000 if J <= 3 else 260_000,
        use_subset=True,
        subset_size=90,
        l2=0.01,
        ftol=1e-7,
        gtol=2e-5,
        do_emission_only_warmstart=True,
        emission_only_maxiter=120 if J <= 3 else 180,
    )

    p_hat, res, is_conv = fit_model_batched(
        J=J,
        Y_stack=Y_m,
        X_stack=X_m,
        Z_stack=Z_m,
        seed=seed,
        sigma_min=0.1,
        sigma_max=3.5,
        print_every=100,
        **cfg,
    )

    ll_total = float(getattr(res, "true_ll", np.nan))
    k_params = len(getattr(res, "x", []))
    bic = np.log(n_obs_m) * k_params - 2.0 * ll_total

    occupancy_min = np.nan
    certainty_mean = np.nan
    if "forward_backward" in globals():
        gammas = []
        for Y_i, X_i, Z_i in zip(data_m.Y, data_m.X, data_m.Z):
            _, log_g = forward_backward(p_hat, Y_i, X_i, Z_i)
            gammas.append(np.exp(log_g))
        gamma_all = np.concatenate(gammas, axis=0)
        occupancy_min = float(gamma_all.mean(axis=0).min())
        certainty_mean = float(gamma_all.max(axis=1).mean())

    interaction_signal = np.nan
    if len(transition_cols) > len(BASE_COLS):
        inter_idx = len(transition_cols) - 1
        interaction_signal = float(np.mean(np.abs(p_hat.beta[:, :, inter_idx])))

    return {
        "model": model_name,
        "J": J,
        "P": len(transition_cols),
        "transition_cols": ", ".join(transition_cols),
        "LL": ll_total,
        "BIC": bic,
        "soft_converged": bool(is_conv),
        "scipy_success": bool(getattr(res, "success", False)),
        "occupancy_min": occupancy_min,
        "certainty_mean": certainty_mean,
        "interaction_signal": interaction_signal,
        "message": str(getattr(res, "message", "")),
    }


all_rows = []
seed_base = 123

for model_idx, (model_name, transition_cols) in enumerate(MODEL_SPECS.items()):
    print(f"\n=== {model_name}: running J=2,3 ===")
    model_rows = []

    for J in J_MIN_SET:
        row = fit_and_score_model(
            model_name=model_name,
            transition_cols=transition_cols,
            J=J,
            seed=seed_base + 17 * model_idx + J,
        )
        model_rows.append(row)
        all_rows.append(row)
        print(f"  J={J} | BIC={row['BIC']:.2f} | soft_conv={row['soft_converged']} | scipy={row['scipy_success']}")

    bic_sorted = sorted([r["BIC"] for r in model_rows if np.isfinite(r["BIC"])])
    bic_gap = bic_sorted[1] - bic_sorted[0] if len(bic_sorted) == 2 else np.inf
    need_j4 = (bic_gap < 6.0) or (not any(r["soft_converged"] for r in model_rows))

    if need_j4:
        print(f"  -> Running J=4 for {model_name} (needed: bic_gap={bic_gap:.2f})")
        row4 = fit_and_score_model(
            model_name=model_name,
            transition_cols=transition_cols,
            J=4,
            seed=seed_base + 17 * model_idx + 4,
        )
        all_rows.append(row4)
        print(f"  J=4 | BIC={row4['BIC']:.2f} | soft_conv={row4['soft_converged']} | scipy={row4['scipy_success']}")

results_ext = pd.DataFrame(all_rows)
display(results_ext.sort_values(["model", "BIC"]).reset_index(drop=True).round(4))

# Best J per model
best_by_model = (
    results_ext.loc[results_ext.groupby("model")["BIC"].idxmin()]
    .sort_values("BIC")
    .reset_index(drop=True)
)

# Convergence stability per model
conv_stability = (
    results_ext.groupby("model")[["soft_converged", "scipy_success"]]
    .mean()
    .rename(columns={
        "soft_converged": "soft_conv_rate",
        "scipy_success": "scipy_success_rate",
    })
    .reset_index()
)

best_by_model = best_by_model.merge(conv_stability, on="model", how="left")

baseline_bic = float(best_by_model.loc[best_by_model["model"] == "Baseline", "BIC"].iloc[0])
best_by_model["delta_BIC_vs_baseline"] = best_by_model["BIC"] - baseline_bic

best_by_model["interpretability_ok"] = (
    (best_by_model["occupancy_min"].fillna(0) >= 0.08)
    & (best_by_model["certainty_mean"].fillna(0) >= 0.55)
)
best_by_model["story_ok"] = best_by_model["interaction_signal"].fillna(0) >= 0.04

print("\n=== Best per model (4 criteria summary) ===")
display(
    best_by_model[[
        "model", "J", "BIC", "delta_BIC_vs_baseline",
        "soft_conv_rate", "scipy_success_rate",
        "occupancy_min", "certainty_mean", "interaction_signal",
        "interpretability_ok", "story_ok",
    ]].round(4)
)

# Keep simplest model that adds real insight
candidate_ext = best_by_model[best_by_model["model"].isin(["Model A", "Model B", "Model C"])].copy()
valid_ext = candidate_ext[
    (candidate_ext["delta_BIC_vs_baseline"] <= -6.0)
    & (candidate_ext["soft_conv_rate"] >= 0.5)
    & (candidate_ext["interpretability_ok"])
    & (candidate_ext["story_ok"])
]

if len(valid_ext) == 0:
    chosen = best_by_model[best_by_model["model"] == "Baseline"].iloc[0]
    print("\nRecommendation: keep Baseline (no single interaction adds clear enough insight).")
else:
    chosen = valid_ext.sort_values("BIC").iloc[0]
    print(f"\nRecommendation: keep {chosen['model']} (J={int(chosen['J'])}) as the simplest insightful extension.")

print(f"Selected model: {chosen['model']} | J={int(chosen['J'])} | BIC={chosen['BIC']:.2f}")
if chosen["model"] != "Baseline":
    print(f"ΔBIC vs baseline: {chosen['delta_BIC_vs_baseline']:.2f}")

best_extension_results = best_by_model.copy()
best_extension_choice = chosen.to_dict()


=== Baseline: running J=2,3 ===
  share_authority_esc computed from decision_episode (mean=0.3508, std=0.0731)
    J=2 start 1/2 iter=100 elapsed=1.8 min
    J=2 start 1/2 iter=200 elapsed=3.0 min
    J=2 start 1/2 iter=300 elapsed=4.2 min
    done: J=2 start 1/2 success=False nit=300 true_negLL=1673.89 msg=STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
    J=2 start 2/2 iter=100 elapsed=1.2 min
    J=2 start 2/2 iter=200 elapsed=2.2 min
    J=2 start 2/2 iter=300 elapsed=3.4 min
    done: J=2 start 2/2 success=False nit=300 true_negLL=1329.96 msg=STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
  J=2 | BIC=3156.31 | soft_conv=False | scipy=False
  share_authority_esc computed from decision_episode (mean=0.3508, std=0.0731)
    J=3 start 1/2 iter=100 elapsed=3.1 min
    J=3 start 1/2 iter=200 elapsed=4.7 min
    J=3 start 1/2 iter=300 elapsed=6.3 min
    done: J=3 start 1/2 success=False nit=300 true_negLL=146.36 msg=STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
    J=3 start 2/2 iter=100 elaps

,model,J,P,transition_cols,LL,BIC,soft_converged,scipy_success,occupancy_min,certainty_mean,interaction_signal,message
0,Baseline,3,4,"within_unit_temporal_benchmark_c, horizontal_p...",2.445410e+01,8.157796e+02,False,False,0.0807,0.9313,NaN,STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
1,Baseline,2,4,"within_unit_temporal_benchmark_c, horizontal_p...",-1.329958e+03,3.156311e+03,False,False,0.1543,0.9973,NaN,STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
2,Baseline,4,4,"within_unit_temporal_benchmark_c, horizontal_p...",-1.000000e+50,2.000000e+50,False,False,0.0807,0.9406,NaN,Time cap reached (12 min)
3,Model A,3,5,"within_unit_temporal_benchmark_c, horizontal_p...",-3.206960e+01,1.000884e+03,False,False,0.0894,0.9383,0.4451,STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
4,Model A,2,5,"within_unit_temporal_benchmark_c, horizontal_p...",-1.287610e+03,3.103640e+03,False,False,0.1539,0.9975,0.0947,STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
5,Model A,4,5,"within_unit_temporal_benchmark_c, horizontal_p...",-1.000000e+50,2.000000e+50,False,False,0.0807,0.9147,0.3614,Time cap reached (12 min)
6,Model B,2,5,"within_unit_temporal_benchmark_c, horizontal_p...",-1.149629e+03,2.827678e+03,False,False,0.1540,0.9976,0.0818,STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
7,Model B,3,5,"within_unit_temporal_benchmark_c, horizontal_p...",-1.000000e+50,2.000000e+50,False,False,0.0904,0.9823,0.3543,Time cap reached (8 min)
8,Model B,4,5,"within_unit_temporal_benchmark_c, horizontal_p...",-1.000000e+50,2.000000e+50,False,False,0.0807,0.9005,0.2266,Time cap reached (12 min)
9,Model C,2,5,"within_unit_temporal_benchmark_c, horizontal_p...",-1.322027e+03,3.172475e+03,True,True,0.1546,0.9970,0.1104,CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*...



=== Best per model (4 criteria summary) ===


,model,J,BIC,delta_BIC_vs_baseline,soft_conv_rate,scipy_success_rate,occupancy_min,certainty_mean,interaction_signal,interpretability_ok,story_ok
0,Baseline,3,815.7796,0.0000,0.0,0.0,0.0807,0.9313,NaN,True,False
1,Model A,3,1000.8842,185.1046,0.0,0.0,0.0894,0.9383,0.4451,True,True
2,Model B,2,2827.6780,2011.8984,0.0,0.0,0.1540,0.9976,0.0818,True,True
3,Model C,2,3172.4746,2356.6950,0.5,0.5,0.1546,0.9970,0.1104,True,True



Recommendation: keep Baseline (no single interaction adds clear enough insight).
Selected model: Baseline | J=3 | BIC=815.78


In [28]:
# ============================================================
# Prep for final refit: enforce selected baseline specification
# ============================================================

transition_cols_final = BASE_COLS

data = load_sequences(DATA_PATH, transition_cols_override=transition_cols_final)
Y_stack = np.stack(data.Y)
X_stack = np.stack(data.X)
Z_stack = np.stack(data.Z)

N, T, D = Y_stack.shape
P = X_stack.shape[2]
K = Z_stack.shape[2]
n_obs_total = N * T

print("Refit prep complete (baseline spec).")
print(f"  transition cols ({P}): {transition_cols_final}")
print(f"  Y_stack={Y_stack.shape}, X_stack={X_stack.shape}, Z_stack={Z_stack.shape}")
print(f"  n_obs_total={n_obs_total}")

  share_authority_esc computed from decision_episode (mean=0.3508, std=0.0731)
Refit prep complete (baseline spec).
  transition cols (4): ['within_unit_temporal_benchmark_c', 'horizontal_peer_benchmark_c', 'threshold_benchmark_c', 'transparency_level_norm']
  Y_stack=(120, 25, 2), X_stack=(120, 25, 4), Z_stack=(120, 25, 8)
  n_obs_total=3000


In [29]:
# Set selected model size from interaction screening decision
best_J = 3
print(f"Using fixed best_J = {best_J} for final refit.")

Using fixed best_J = 3 for final refit.


#### Final refit with best J!!!

In [27]:
# ============================================================
# STAGE 2: FINAL REFIT (robust version)
# ============================================================

import time
import numpy as np

# ----------------------------
# Select best J from Stage 1
# ----------------------------
if "best_J" in globals():
    BEST_J = int(best_J)
elif "best_J_screen" in globals():
    BEST_J = int(best_J_screen)
else:
    raise NameError("Run Stage 1 screening first.")

print(f"\n=== STAGE 2: FINAL REFIT (BEST J={BEST_J}) ===")

# ----------------------------
# Warm start
# Skip any warm start whose W shape doesn't match Z_stack K dimension
# ----------------------------
warm_list = []
K_now = Z_stack.shape[2]

for src_name, src_dict in [
    ("best_params_full_by_J", globals().get("best_params_full_by_J", {})),
    ("warm_by_J", globals().get("warm_by_J", {})),
]:
    if isinstance(src_dict, dict) and BEST_J in src_dict:
        ws = src_dict[BEST_J]
        if hasattr(ws, "W") and ws.W.shape[2] == K_now:
            warm_list.append(ws)
            print(f"✓ Using warm start from {src_name} (K={ws.W.shape[2]})")
            break
        else:
            ws_k = ws.W.shape[2] if hasattr(ws, "W") else "?"
            print(f"⚠️ Skipping warm start from {src_name} (K={ws_k} ≠ {K_now})")

if not warm_list:
    print("⚠️ No compatible warm start found — random initialization")

# ----------------------------
# Stronger final configuration
# NOTE: l2=0.01 prevents degenerate absorbing transitions
#       Caps scale with J: J=4 needs more iterations and time to converge.
# ----------------------------
final_cfg = dict(
    maxiter=1600 if BEST_J <= 3 else 2200,
    n_starts=12 if BEST_J <= 3 else 15,
    seed=777,
    time_cap_min=45 if BEST_J <= 3 else 90,
    diag_bias=2.4 if BEST_J <= 3 else 2.7,
    maxfun=900_000 if BEST_J <= 3 else 1_300_000,
    use_subset=False,
    l2=0.01,
    do_emission_only_warmstart=True,
    emission_only_maxiter=300 if BEST_J <= 3 else 400,
    ftol=1e-9,
    gtol=1e-6,
)

# ----------------------------
# Run final estimation
# ----------------------------
t0 = time.time()

best_p_final, best_res_final, best_is_conv_final = fit_model_batched(
    J=BEST_J,
    Y_stack=Y_stack,
    X_stack=X_stack,
    Z_stack=Z_stack,
    sigma_min=0.1,
    sigma_max=3.5,
    print_every=50,
    warm_starts=warm_list,
    **final_cfg
)

elapsed = time.time() - t0

# ----------------------------
# Metrics
# ----------------------------
ll_total = float(getattr(best_res_final, "true_ll", np.nan))
k_params = len(getattr(best_res_final, "x", []))
bic = np.log(n_obs_total) * k_params - 2.0 * ll_total
aic = 2.0 * k_params - 2.0 * ll_total

print("\n" + "="*60)
print(f"FINAL MODEL (J={BEST_J})")
print(f"LL:  {ll_total:.2f}")
print(f"AIC: {aic:.2f}")
print(f"BIC: {bic:.2f}")
print(f"Soft-converged: {bool(best_is_conv_final)}")
print(f"SciPy success:  {bool(getattr(best_res_final,'success',False))}")
print(f"Runtime: {elapsed/60:.1f} minutes")
print("="*60)

best_model = best_p_final



=== STAGE 2: FINAL REFIT (BEST J=3) ===
⚠️ No compatible warm start found — random initialization
    J=3 start 1/12 iter=50 elapsed=0.9 min
    J=3 start 1/12 iter=100 elapsed=1.9 min
    J=3 start 1/12 iter=150 elapsed=2.7 min
    J=3 start 1/12 iter=200 elapsed=3.4 min
    J=3 start 1/12 iter=250 elapsed=4.4 min
    J=3 start 1/12 iter=300 elapsed=5.2 min
    J=3 start 1/12 iter=350 elapsed=5.9 min
    J=3 start 1/12 iter=400 elapsed=6.7 min
    J=3 start 1/12 iter=450 elapsed=7.5 min
    J=3 start 1/12 iter=500 elapsed=8.3 min
    J=3 start 1/12 iter=550 elapsed=9.0 min
    J=3 start 1/12 iter=600 elapsed=9.8 min
    J=3 start 1/12 iter=650 elapsed=10.6 min
    J=3 start 1/12 iter=700 elapsed=11.3 min
    J=3 start 1/12 iter=750 elapsed=12.1 min
    J=3 start 1/12 iter=800 elapsed=12.9 min
    J=3 start 1/12 iter=850 elapsed=13.7 min
    J=3 start 1/12 iter=900 elapsed=14.4 min
    J=3 start 1/12 iter=950 elapsed=15.1 min
    J=3 start 1/12 iter=1000 elapsed=15.9 min
    J=3 start

In [30]:

import pickle

# Save best model artifacts for later reuse (no refit needed)

best_J = int(best_J) if "best_J" in globals() else int(globals().get("BEST_J", best_J_screen))

model_artifacts = {
    "best_model": best_model,
    "best_J": int(best_J),
    "best_res_final": globals().get("best_res_final", None),
    "final_cfg": globals().get("final_cfg", None),
    "ll_total": globals().get("ll_total", None),
    "k_params": globals().get("k_params", None),
    "n_obs_total": globals().get("n_obs_total", None),
    "aic": globals().get("aic", None),
    "bic": globals().get("bic", None),
    "emission_cols": globals().get("emission_cols", None),
    "transition_cols": globals().get("transition_cols", None),
    "control_cols": globals().get("control_cols", None),
    "label_map": globals().get("label_map", None),
    "state_order_1idx": globals().get("state_order_1idx", None),
    "y_scaler": getattr(data, "y_scaler", None),
    "x_scaler": getattr(data, "x_scaler", None),
    "z_scaler": getattr(data, "z_scaler", None),
}

out_path = Path("best_model_artifacts_dataset2_2emissions_7covariates.pkl")
with out_path.open("wb") as f:
    pickle.dump(model_artifacts, f)

print(f"Saved best model artifacts → {out_path.resolve()}")


Saved best model artifacts → C:\Users\Admin\OneDrive\Desktop\Algorithm-Appreciation-and-Aversion-in-Triadic-Delegation-Settings\data_analysis\best_model_artifacts_dataset2_2emissions_7covariates.pkl
